# Tutoriel interactif — `linalg_pedagopy` v3.0

Ce notebook propose une prise en main complète et interactive du package **`linalg_pedagopy` v3.0**.
Il est fondé sur l'ouvrage *Introduction to Applied Linear Algebra — Vectors, Matrices, and Least Squares (VMLS)* de Stephen Boyd et Lieven Vandenberghe.

---

### Objectifs d'apprentissage :
1. **Comprendre le calcul formel exact sur $\mathbb{R}$** sans perte de précision, avec l'oracle de nullité à 4 verdicts (`ZeroOracle`).
2. **Manipuler les vecteurs et matrices immuables** (`Vector`, `Matrix`) et leurs métriques pédagogiques (RMS, moyenne, centrage, écart-type VMLS, angle avec Cauchy-Schwarz).
3. **Explorer les 4 points de vue du produit matriciel** (coefficients, lignes, colonnes, produits extérieurs).
4. **Résoudre des systèmes linéaires** par réduction de Gauss-Jordan (`GaussJordan`), avec choix du pivot le plus simple et extraction d'ensembles de solutions typés (`UniqueSolution`, `AffineSolution`, `EmptySolution`).
5. **Maîtriser la discussion automatique selon des paramètres** (`CaseSplit`).
6. **Effectuer des résolutions triangulaires** par substitution arrière (`BackSubstitution`) et avant (`ForwardSubstitution`).
7. **Orthogonaliser des familles de vecteurs** par Gram-Schmidt rationnel ou normalisé (`GramSchmidt`), détecter les liaisons linéaires avec témoin exact, et factoriser sous forme $A = QR$ (`QR`).
8. **Résoudre des problèmes de moindres carrés** (`LeastSquares`) en arithmétique rationnelle exacte et vérifier la condition géométrique $A^T \hat{r} = 0$.
9. **Tester les limites du calcul flottant machine** face au calcul exact avec le module `compare` (matrice de Läuchli, conditionnement de Hilbert).
10. **Pratiquer sur les exercices du recueil officiel de VMLS** grâce au **Tuteur interactif** (`tuteur`).

## 1. Importation et Configuration du Package

Nous importons les composants de `linalg_pedagopy` ainsi que les utilitaires d'affichage riche de Jupyter (`display`, `Markdown`, `Latex`).
Dans Jupyter, `display(...)` combiné aux objets du package active le rendu natif en **Markdown** et **LaTeX MathJax**.

In [1]:
import sys
from pathlib import Path
from fractions import Fraction
import sympy as sp
from IPython.display import display, Markdown, Latex

# Configuration du chemin d'accès vers src/ si non installé en mode éditable
src_path = str(Path.cwd() / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from linalg_pedagopy import (
    Vector, Matrix, symbols, to_exact,
    GaussJordan, BackSubstitution, ForwardSubstitution,
    GramSchmidt, QR, LeastSquares,
    tracing, section, record, config, using_config,
    compare
)
from linalg_pedagopy.tuteur import exercice, glossaire

display(Markdown(f"**`linalg_pedagopy` initialisé avec succès !** Version / État : `{config.state}`"))

**`linalg_pedagopy` initialisé avec succès !** Version / État : `ConfigState(float_policy='rationalize', locale='fr', default_trace_level=1, max_case_depth=3, max_cases=8, tutor_mode='etudiant', hint_min_attempts=0, hint_min_delay_seconds=0, journal_path=WindowsPath('.linalg_pedagopy_journal.json'))`

## 2. Vecteurs et Métriques de VMLS (`Vector`)

Les objets `Vector` représentent des vecteurs colonnes $n \times 1$ strictement **immuables** à coefficients exacts.
Ils s'affichent automatiquement sous forme de matrice colonne LaTeX MathJax dans Jupyter.

In [2]:
# 2.1 Construction de vecteurs à coefficients exacts (fractions, radicaux)
v1 = Vector([1, "1/3", Fraction(5, 2), sp.sqrt(2)])

display(Markdown(r"**Vecteur $v_1$ (rendu LaTeX MathJax) :**"))
display(v1)
display(Markdown(f"- **Taille :** {len(v1)}\n- **Forme matricielle :** `{v1.shape}`"))

# Vecteurs canoniques et particuliers
zero_3 = Vector.zeros(3)
ones_3 = Vector.ones(3)
e2 = Vector.unit(1, 4)  # 2e vecteur canonique dans R^4 (indice 0-based)

display(Markdown(r"**Vecteur canonique $e_2 \in \mathbb{R}^4$ :**"))
display(e2)

**Vecteur $v_1$ (rendu LaTeX MathJax) :**

Vector([1, 1/3, 5/2, sqrt(2)])

- **Taille :** 4
- **Forme matricielle :** `(4, 1)`

**Vecteur canonique $e_2 \in \mathbb{R}^4$ :**

Vector([0, 1, 0, 0])

In [3]:
# 2.2 Opérations algébriques et combinaisons linéaires
a = Vector([1, 2, 3])
b = Vector([4, -1, 2])

display(Markdown(r"**Addition $a + b$ :**"))
display(a + b)

display(Markdown(r"**Multiplication par un scalaire $3a$ :**"))
display(3 * a)

# Combinaison linéaire tracée : 2*a - 3*b
comb = Vector.combination([a, b], [2, -3])
display(Markdown(r"**Combinaison linéaire $2a - 3b$ :**"))
display(comb)

**Addition $a + b$ :**

Vector([5, 1, 5])

**Multiplication par un scalaire $3a$ :**

Vector([3, 6, 9])

**Combinaison linéaire $2a - 3b$ :**

Vector([-10, 7, 0])

In [4]:
# 2.3 Métriques statistiques et géométriques de VMLS (ch. 1 & 3)
x = Vector([1, -2, 3, 2])
y = Vector([2, 1, 0, 1])

display(Markdown(f"""### Métriques statistiques de $x$ (VMLS ch. 1 & 3)
- **Moyenne arithmétique $\\mathrm{{avg}}(x)$ :** ${sp.latex(x.avg())}$
- **Vecteur centré $\\tilde{{x}} = \\mathrm{{demean}}(x)$ :**"""))
display(x.demean())

display(Markdown(f"""- **Norme euclidienne $\\|x\\|$ :** ${sp.latex(x.norm())}$
- **Moyenne quadratique $\\mathrm{{rms}}(x)$ :** ${sp.latex(x.rms())}$
- **Écart-type VMLS $\\mathrm{{std}}(x)$ :** ${sp.latex(x.std())}$

### Relations géométriques entre $x$ et $y$
- **Produit scalaire $x^T y$ :** ${sp.latex(x.dot(y))}$
- **Distance $\\|x - y\\|$ :** ${sp.latex(x.distance(y))}$
- **Angle $\\theta$ :** ${sp.latex(x.angle(y))} \\text{{ rad}}$
- **Coefficient de corrélation $\\rho(x, y)$ :** ${sp.latex(x.correlation(y))}$"""))

### Métriques statistiques de $x$ (VMLS ch. 1 & 3)
- **Moyenne arithmétique $\mathrm{avg}(x)$ :** $1$
- **Vecteur centré $\tilde{x} = \mathrm{demean}(x)$ :**

Vector([0, -3, 2, 1])

- **Norme euclidienne $\|x\|$ :** $3 \sqrt{2}$
- **Moyenne quadratique $\mathrm{rms}(x)$ :** $\frac{3 \sqrt{2}}{2}$
- **Écart-type VMLS $\mathrm{std}(x)$ :** $\frac{\sqrt{14}}{2}$

### Relations géométriques entre $x$ et $y$
- **Produit scalaire $x^T y$ :** $2$
- **Distance $\|x - y\|$ :** $2 \sqrt{5}$
- **Angle $\theta$ :** $\operatorname{acos}{\left(\frac{\sqrt{3}}{9} \right)} \text{ rad}$
- **Coefficient de corrélation $\rho(x, y)$ :** $- \frac{\sqrt{7}}{7}$

## 3. Matrices et les 4 Points de Vue du Produit Matriciel (`Matrix`)

Dans le chapitre 10 de VMLS, le produit de matrices $A B$ est interprété de 4 manières équivalentes et complémentaires :
1. **Par coefficients** (`entries`) : produits scalaires ligne · colonne.
2. **Par lignes** (`rows`) : chaque ligne de $A B$ est combinaison linéaire des lignes de $B$.
3. **Par colonnes** (`columns`) : chaque colonne de $A B$ est combinaison linéaire des colonnes de $A$.
4. **Par produits extérieurs** (`outer`) : somme de $k$ matrices de rang 1 $a_i b_i^T$.

In [5]:
# 3.1 Création et produit de matrices
A = Matrix([
    [1, 2],
    [3, 4],
    [5, 6]
])

B = Matrix([
    [2, -1],
    [0, 3]
])

display(Markdown(r"**Matrice $A$ ($3 \times 2$) :**"))
display(A)
display(Markdown(r"**Matrice $B$ ($2 \times 2$) :**"))
display(B)
display(Markdown(r"**Produit matriciel $A B$ ($3 \times 2$) :**"))
display(A @ B)

**Matrice $A$ ($3 \times 2$) :**

Matrix([[1, 2], [3, 4], [5, 6]])

**Matrice $B$ ($2 \times 2$) :**

Matrix([[2, -1], [0, 3]])

**Produit matriciel $A B$ ($3 \times 2$) :**

Matrix([[2, 5], [6, 9], [10, 13]])

In [6]:
# 3.2 Visualisation des 4 vues pédagogiques du produit matriciel
for vue in ("entries", "rows", "columns", "outer"):
    with tracing(level=2) as t:
        _ = A.matmul(B, view=vue)
    display(Markdown(f"### Vue : `{vue}`"))
    display(t.tree)

### Vue : `entries`

- Produit matriciel (3×2) @ (2×2) — vue par coefficients (produits scalaires ligne·colonne).
  - Résultat du produit matriciel : [(2 5); (6 9); (10 13)].
    $$\left[\begin{array}{cc} 2 & 5 \\ 6 & 9 \\ 10 & 13 \end{array}\right]$$

### Vue : `rows`

- Produit matriciel (3×2) @ (2×2) — vue par lignes (combinaisons linéaires des lignes de B).
  - Résultat du produit matriciel : [(2 5); (6 9); (10 13)].
    $$\left[\begin{array}{cc} 2 & 5 \\ 6 & 9 \\ 10 & 13 \end{array}\right]$$

### Vue : `columns`

- Produit matriciel (3×2) @ (2×2) — vue par colonnes (combinaisons linéaires des colonnes de A).
  - Résultat du produit matriciel : [(2 5); (6 9); (10 13)].
    $$\left[\begin{array}{cc} 2 & 5 \\ 6 & 9 \\ 10 & 13 \end{array}\right]$$

### Vue : `outer`

- Produit matriciel (3×2) @ (2×2) — vue en somme de produits extérieurs (VMLS ch. 10).
  - Résultat du produit matriciel : [(2 5); (6 9); (10 13)].
    $$\left[\begin{array}{cc} 2 & 5 \\ 6 & 9 \\ 10 & 13 \end{array}\right]$$

In [7]:
# 3.3 Transposée, Matrice de Gram et Blocs
display(Markdown(r"**Transposée $A^T$ :**"))
display(A.T)

# Matrice de Gram G = A^T A (VMLS ch. 10)
G = A.gram()
display(Markdown(r"**Matrice de Gram $G = A^T A$ :**"))
display(G)

# Matrice par blocs
M_bloc = Matrix.block([
    [B, Matrix.zeros(2, 2)],
    [Matrix.identity(2), B]
])
display(Markdown(r"**Matrice par blocs $4 \times 4$ :**"))
display(M_bloc)

**Transposée $A^T$ :**

Matrix([[1, 3, 5], [2, 4, 6]])

**Matrice de Gram $G = A^T A$ :**

Matrix([[35, 44], [44, 56]])

**Matrice par blocs $4 \times 4$ :**

Matrix([[2, -1, 0, 0], [0, 3, 0, 0], [1, 0, 2, -1], [0, 1, 0, 3]])

## 4. Systèmes Linéaires et Élimination de Gauss-Jordan (`GaussJordan`)

`GaussJordan` réduit une matrice augmentée $[A \mid b]$ à sa forme échelonnée réduite, avec :
- Le choix du **pivot le plus simple** (±1, petit entier, rationnel de petite hauteur) (**D11**) ;
- Des résultats rigoureusement typés : `UniqueSolution`, `AffineSolution` ou `EmptySolution` (**D8**) ;
- Des vérifications automatiques *a posteriori* (`res.verify()`).

In [8]:
# 4.1 Système avec solution unique
A = Matrix([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])
b = Vector([8, -11, -3])

res_gj = GaussJordan(A, b).run()

display(Markdown(f"""### Résultat de Gauss-Jordan :
- **Rang de $A$ :** {res_gj.rank}
- **Colonnes pivots :** {res_gj.pivots}"""))
display(res_gj.solution)

display(Markdown("### Trace complète de la réduction (niveau 2 — pas à pas) :"))
display(res_gj.explain(level=2))

display(Markdown("### Contrôle a posteriori :"))
display(res_gj.verify())

### Résultat de Gauss-Jordan :
- **Rang de $A$ :** 3
- **Colonnes pivots :** (0, 1, 2)

**Solution unique :**

$$\left[\begin{array}{c} 2 \\ 3 \\ -1 \end{array}\right]$$

### Trace complète de la réduction (niveau 2 — pas à pas) :

- Réduction de Gauss-Jordan d'une matrice 3×3.
  $$\left[\begin{array}{ccc|c} 2 & 1 & -1 & 8 \\ -3 & -1 & 2 & -11 \\ -2 & 1 & 2 & -3 \end{array}\right]$$
  - Traitement de la colonne 1 (ligne pivot courante : 1).
    - Choix du pivot en (1,1) de valeur 2 (stratégie : simplest).
    - On normalise le pivot de la ligne 1 : L1 ← (1/2)·L1.
      $$\xrightarrow{L_{1} \leftarrow \left(\frac{1}{2}\right) L_{1}} \left[\begin{array}{ccc|c} \boxed{1} & \color{blue}{\frac{1}{2}} & \color{blue}{- \frac{1}{2}} & \color{blue}{4} \\ -3 & -1 & 2 & -11 \\ -2 & 1 & 2 & -3 \end{array}\right]$$
    - On annule le coefficient (2,1) : L2 ← L2 + 3·L1.
      $$\xrightarrow{L_{2} \leftarrow L_{2} + \left(3\right) L_{1}} \left[\begin{array}{ccc|c} \boxed{1} & \frac{1}{2} & - \frac{1}{2} & 4 \\ \color{blue}{0} & \color{blue}{\frac{1}{2}} & \color{blue}{\frac{1}{2}} & \color{blue}{1} \\ -2 & 1 & 2 & -3 \end{array}\right]$$
    - On annule le coefficient (3,1) : L3 ← L3 + 2·L1.
      $$\xrightarrow{L_{3} \leftarrow L_{3} + \left(2\right) L_{1}} \left[\begin{array}{ccc|c} \boxed{1} & \frac{1}{2} & - \frac{1}{2} & 4 \\ 0 & \frac{1}{2} & \frac{1}{2} & 1 \\ \color{blue}{0} & \color{blue}{2} & \color{blue}{1} & \color{blue}{5} \end{array}\right]$$
  - Traitement de la colonne 2 (ligne pivot courante : 2).
    - Choix du pivot en (3,2) de valeur 2 (stratégie : simplest).
    - On échange les lignes L2 et L3 : L2 ↔ L3.
      $$\xrightarrow{L_{2} \leftrightarrow L_{3}} \left[\begin{array}{ccc|c} 1 & \frac{1}{2} & - \frac{1}{2} & 4 \\ \color{blue}{0} & \boxed{2} & \color{blue}{1} & \color{blue}{5} \\ \color{blue}{0} & \color{blue}{\frac{1}{2}} & \color{blue}{\frac{1}{2}} & \color{blue}{1} \end{array}\right]$$
    - On normalise le pivot de la ligne 2 : L2 ← (1/2)·L2.
      $$\xrightarrow{L_{2} \leftarrow \left(\frac{1}{2}\right) L_{2}} \left[\begin{array}{ccc|c} 1 & \frac{1}{2} & - \frac{1}{2} & 4 \\ \color{blue}{0} & \boxed{1} & \color{blue}{\frac{1}{2}} & \color{blue}{\frac{5}{2}} \\ 0 & \frac{1}{2} & \frac{1}{2} & 1 \end{array}\right]$$
    - On annule le coefficient (1,2) : L1 ← L1 - 1/2·L2.
      $$\xrightarrow{L_{1} \leftarrow L_{1} - \left(\frac{1}{2}\right) L_{2}} \left[\begin{array}{ccc|c} \color{blue}{1} & \color{blue}{0} & \color{blue}{- \frac{3}{4}} & \color{blue}{\frac{11}{4}} \\ 0 & \boxed{1} & \frac{1}{2} & \frac{5}{2} \\ 0 & \frac{1}{2} & \frac{1}{2} & 1 \end{array}\right]$$
    - On annule le coefficient (3,2) : L3 ← L3 - 1/2·L2.
      $$\xrightarrow{L_{3} \leftarrow L_{3} - \left(\frac{1}{2}\right) L_{2}} \left[\begin{array}{ccc|c} 1 & 0 & - \frac{3}{4} & \frac{11}{4} \\ 0 & \boxed{1} & \frac{1}{2} & \frac{5}{2} \\ \color{blue}{0} & \color{blue}{0} & \color{blue}{\frac{1}{4}} & \color{blue}{- \frac{1}{4}} \end{array}\right]$$
  - Traitement de la colonne 3 (ligne pivot courante : 3).
    - Choix du pivot en (3,3) de valeur 1/4 (stratégie : simplest).
    - On normalise le pivot de la ligne 3 : L3 ← (4)·L3.
      $$\xrightarrow{L_{3} \leftarrow \left(4\right) L_{3}} \left[\begin{array}{ccc|c} 1 & 0 & - \frac{3}{4} & \frac{11}{4} \\ 0 & 1 & \frac{1}{2} & \frac{5}{2} \\ \color{blue}{0} & \color{blue}{0} & \boxed{1} & \color{blue}{-1} \end{array}\right]$$
    - On annule le coefficient (1,3) : L1 ← L1 + 3/4·L3.
      $$\xrightarrow{L_{1} \leftarrow L_{1} + \left(\frac{3}{4}\right) L_{3}} \left[\begin{array}{ccc|c} \color{blue}{1} & \color{blue}{0} & \color{blue}{0} & \color{blue}{2} \\ 0 & 1 & \frac{1}{2} & \frac{5}{2} \\ 0 & 0 & \boxed{1} & -1 \end{array}\right]$$
    - On annule le coefficient (2,3) : L2 ← L2 - 1/2·L3.
      $$\xrightarrow{L_{2} \leftarrow L_{2} - \left(\frac{1}{2}\right) L_{3}} \left[\begin{array}{ccc|c} 1 & 0 & 0 & 2 \\ \color{blue}{0} & \color{blue}{1} & \color{blue}{0} & \color{blue}{3} \\ 0 & 0 & \boxed{1} & -1 \end{array}\right]$$
  - Pivots non nuls, rang 3, solution unique : x = ( 2, 3, −1 ).
    $$\left[\begin{array}{c} 2 \\ 3 \\ -1 \end{array}\right]$$
  - Forme échelonnée réduite atteinte : rang = 3, colonnes pivots = (1, 2, 3).
    $$\left[\begin{array}{ccc|c} 1 & 0 & 0 & 2 \\ 0 & 1 & 0 & 3 \\ 0 & 0 & 1 & -1 \end{array}\right]$$

### Contrôle a posteriori :

- Vérification a posteriori du résultat de Gauss-Jordan.
  - Contrôle vérifié : A x - b = (0, 0, 0).

In [9]:
# 4.2 Système sous-déterminé (Ensemble affine de solutions)
A_aff = Matrix([[1, 2, -1], [2, 4, -2]])
b_aff = Vector([3, 6])

res_aff = GaussJordan(A_aff, b_aff).run()

display(Markdown(f"**Type de solution :** `{type(res_aff.solution).__name__}`"))
display(res_aff.solution)

# Évaluation d'une solution pour des paramètres donnés t1=2, t2=1
x_part = res_aff.solution.at_params(2, 1)
display(Markdown(r"**Solution particulière pour $t_1 = 2, t_2 = 1$ :**"))
display(x_part)
ok_part = (A_aff @ x_part == b_aff)
display(Markdown(f"- **Vérification $A x_\\text{{part}} == b$ :** `{ok_part}`"))

**Type de solution :** `AffineSolution`

**Ensemble affine de solutions (2 variable(s) libre(s)) :**

$$x = \left[\begin{array}{c} 3 \\ 0 \\ 0 \end{array}\right] + t_1 \left[\begin{array}{c} -2 \\ 1 \\ 0 \end{array}\right] + t_2 \left[\begin{array}{c} 1 \\ 0 \\ 1 \end{array}\right]$$

**Solution particulière pour $t_1 = 2, t_2 = 1$ :**

Vector([0, 2, 1])

- **Vérification $A x_\text{part} == b$ :** `True`

In [10]:
# 4.3 Système incompatible (Ensemble vide)
A_inc = Matrix([[1, 2], [2, 4]])
b_inc = Vector([3, 7])

res_inc = GaussJordan(A_inc, b_inc).run()
display(res_inc.solution)

**Système incompatible :** ensemble vide $\emptyset$.

- Ligne $L_{2}$ : $0 = 1$ (témoin : $0 = 1$).

In [11]:
# 4.4 Inversion formelle d'une matrice
A_inv = Matrix([[1, 2], [3, 5]])
inv_result = A_inv.inv()

display(Markdown(r"**Matrice initiale $A$ :**"))
display(A_inv)
display(Markdown(r"**Matrice inverse $A^{-1}$ :**"))
display(inv_result)
is_id = (A_inv @ inv_result == Matrix.identity(2))
display(Markdown(f"- **Vérification $A A^{{-1}} == I_2$ :** `{is_id}`"))

**Matrice initiale $A$ :**

Matrix([[1, 2], [3, 5]])

**Matrice inverse $A^{-1}$ :**

Matrix([[-5, 2], [3, -1]])

- **Vérification $A A^{-1} == I_2$ :** `True`

## 5. Discussion Automatique selon des Paramètres (`CaseSplit`)

Lorsqu'un pivot ou un test de compatibilité dépend d'un paramètre, l'algorithme produit un objet `CaseSplit`
qui distingue automatiquement les cas génériques et les cas particuliers (**D4**).

In [12]:
# Déclaration d'un paramètre formel réel 'a'
a, = symbols("a")

A_param = Matrix([
    [1, 1, 1],
    [1, a, 1],
    [1, 1, a]
])
b_param = Vector([1, 2, 3])

cs = GaussJordan(A_param, b_param).run()

# Rendu Markdown automatique de la discussion par cas
display(cs)

### Discussion selon $a$

- **Cas $a \neq 1$ (générique)** : pivots non nuls, rang 3, solution unique
  - x = ( (a − 4)/(a − 1), 1/(a − 1), 2/(a − 1) )
- **Cas $a = 1$** : L2 et L3 deviennent (0 0 0 | 1) et (0 0 0 | 2)
  - système incompatible (témoin : 0 = 1)

In [13]:
# Spécialisation numérique du CaseSplit
# Pour a = 2 (branche générique a ≠ 1)
sol_a2 = cs.at(a=2)
display(Markdown(r"**Pour $a = 2$ (branche générique $a \neq 1$) :**"))
display(sol_a2.solution)

# Pour a = 1 (branche particulière d'incompatibilité)
sol_a1 = cs.at(a=1)
display(Markdown(r"**Pour $a = 1$ (branche particulière) :**"))
display(sol_a1.solution)

**Pour $a = 2$ (branche générique $a \neq 1$) :**

**Solution unique :**

$$\left[\begin{array}{c} -2 \\ 1 \\ 2 \end{array}\right]$$

**Pour $a = 1$ (branche particulière) :**

**Système incompatible :** ensemble vide $\emptyset$.

- Ligne $L_{2}$ : $0 = 1$ (témoin : $0 = 1$).

## 6. Substitutions Triangulaires (`BackSubstitution`, `ForwardSubstitution`)

VMLS résout les systèmes échelonnés triangulaires sans jamais inverser de matrice,
via une remontée (`BackSubstitution`) ou une descente (`ForwardSubstitution`) tracée pas à pas.

In [14]:
# Résolution de R x = c avec R triangulaire supérieure
R = Matrix([
    [2, -1, 3],
    [0, 4, 2],
    [0, 0, -2]
])
c = Vector([9, 8, -6])

res_back = BackSubstitution(R, c).run()

display(Markdown(r"**Solution $x$ :**"))
display(res_back.x)
display(Markdown("### Trace pas à pas de la substitution arrière :"))
display(res_back)

**Solution $x$ :**

Vector([1/4, 1/2, 3])

### Trace pas à pas de la substitution arrière :

- Résolution par substitution arrière de R x = c (taille 3×3).
  $$\left[\begin{array}{ccc} 2 & -1 & 3 \\ 0 & 4 & 2 \\ 0 & 0 & -2 \end{array}\right]$$
  - Calcul de x_3 : x_3 = (−6 - (0)) / (−2) = 3.
  - Calcul de x_2 : x_2 = (8 - (6)) / (4) = 1/2.
  - Calcul de x_1 : x_1 = (9 - (17/2)) / (2) = 1/4.
  - Solution du système triangulaire : x = ( 1/4, 1/2, 3 ).
    $$\left[\begin{array}{c} \frac{1}{4} \\ \frac{1}{2} \\ 3 \end{array}\right]$$

## 7. Orthogonalisation de Gram-Schmidt et Factorisation QR

Deux variantes de Gram-Schmidt sont implémentées :
1. `normalize="end"` (défaut) : orthogonalisation rationnelle $\tilde{q}_i = a_i - \sum \tilde{r}_{ji} \tilde{q}_j$, entièrement dans $\mathbb{Q}$.
2. `normalize="each"` : algorithme classique de VMLS qui normalise à chaque étape.

In [15]:
# 7.1 Gram-Schmidt rationnel et normalisé
a1 = Vector([1, 1, 0])
a2 = Vector([1, 0, 1])
a3 = Vector([0, 1, 1])

gs = GramSchmidt([a1, a2, a3], normalize="end")
res_gs = gs.run()

display(Markdown(f"""### Orthogonalisation de Gram-Schmidt :
- **Famille libre :** `{res_gs.independent}`
- **Vecteurs orthogonaux rationnels $\\tilde{{q}}_i$ :**"""))
for i, q_t in enumerate(res_gs.orthogonal, 1):
    display(Markdown(f"$$\\tilde{{q}}_{{{i}}} = {q_t._repr_latex_().strip('$')}$$"))

display(Markdown("- **Vecteurs orthonormés finaux $q_i$ :**"))
for i, q in enumerate(res_gs.orthonormal, 1):
    display(Markdown(f"$$q_{{{i}}} = {q._repr_latex_().strip('$')}$$"))

### Orthogonalisation de Gram-Schmidt :
- **Famille libre :** `True`
- **Vecteurs orthogonaux rationnels $\tilde{q}_i$ :**

$$\tilde{q}_{1} = \left[\begin{array}{c} 1 \\ 1 \\ 0 \end{array}\right]$$

$$\tilde{q}_{2} = \left[\begin{array}{c} \frac{1}{2} \\ - \frac{1}{2} \\ 1 \end{array}\right]$$

$$\tilde{q}_{3} = \left[\begin{array}{c} - \frac{2}{3} \\ \frac{2}{3} \\ \frac{2}{3} \end{array}\right]$$

- **Vecteurs orthonormés finaux $q_i$ :**

$$q_{1} = \left[\begin{array}{c} \frac{\sqrt{2}}{2} \\ \frac{\sqrt{2}}{2} \\ 0 \end{array}\right]$$

$$q_{2} = \left[\begin{array}{c} \frac{\sqrt{6}}{6} \\ - \frac{\sqrt{6}}{6} \\ \frac{\sqrt{6}}{3} \end{array}\right]$$

$$q_{3} = \left[\begin{array}{c} - \frac{\sqrt{3}}{3} \\ \frac{\sqrt{3}}{3} \\ \frac{\sqrt{3}}{3} \end{array}\right]$$

In [16]:
# 7.2 Détection automatique de dépendance linéaire et témoin exact
v1 = Vector([1, 2, -1])
v2 = Vector([2, 0, 1])
v3 = 2 * v1 - 3 * v2  # Combinaison linéaire

res_dep = GramSchmidt([v1, v2, v3], strict=False).run()
display(Markdown(f"""### Famille liée détectée :
- **Famille libre ?** `{res_dep.independent}`
- **Indice du premier vecteur lié :** $a_{{{res_dep.dependent_index + 1}}}$
- **Témoin exact d'interdépendance :** ${sp.latex(res_dep.witness)}$"""))

### Famille liée détectée :
- **Famille libre ?** `False`
- **Indice du premier vecteur lié :** $a_{3}$
- **Témoin exact d'interdépendance :** $\mathtt{\text{a\_3 = (3/2)·q̃\_1 + (-3)·q̃\_2}}$

In [17]:
# 7.3 Factorisation QR complète : A = QR
A_mat = Matrix([
    [1, 1],
    [1, 0],
    [0, 1]
])

res_qr = QR(A_mat).run()

display(Markdown(r"**Matrice $Q$ (colonnes orthonormées) :**"))
display(res_qr.Q)
display(Markdown(r"**Matrice $R$ (triangulaire supérieure à diagonale $> 0$) :**"))
display(res_qr.R)
display(Markdown(r"**Matrice $\tilde{Q}$ rationnelle :**"))
display(res_qr.Q_tilde)

display(Markdown(r"### Contrôle a posteriori ($Q^T Q = I_2$ et $Q R = A$) :"))
display(res_qr.verify())

**Matrice $Q$ (colonnes orthonormées) :**

Matrix([[sqrt(2)/2, sqrt(6)/6], [sqrt(2)/2, -sqrt(6)/6], [0, sqrt(6)/3]])

**Matrice $R$ (triangulaire supérieure à diagonale $> 0$) :**

Matrix([[sqrt(2), sqrt(2)/2], [0, sqrt(6)/2]])

**Matrice $\tilde{Q}$ rationnelle :**

Matrix([[1, 1/2], [1, -1/2], [0, 1]])

### Contrôle a posteriori ($Q^T Q = I_2$ et $Q R = A$) :

- Vérification de la factorisation QR : contrôle de Q R = A et Q^T Q = I_2 via l'oracle.
  - Contrôle Q R - A = 0 : vérifié coefficient par coefficient.
  - Contrôle Q^T Q - I_2 = 0 : vérifié coefficient par coefficient.

## 8. Moindres Carrés Linéaires (`LeastSquares`)

Pour un problème surdéterminé $A x \approx b$ ($m \ge n$ avec $A$ de rang plein en colonnes),
`LeastSquares` résout $\min \|A x - b\|^2$.

Par défaut, la méthode `"rational_qr"` résout $\tilde{R} \hat{x} = D^{-2} \tilde{Q}^T b$ **entièrement dans $\mathbb{Q}$ sans aucune racine carrée**.

In [18]:
A = Matrix([
    [1, 0],
    [1, 1],
    [1, 2],
    [1, 3]
])
b = Vector([1, 2, 2, 4])

ls = LeastSquares(A, b, method="rational_qr").run()

display(Markdown(r"### Solution des Moindres Carrés :"))
display(Markdown(r"- **Solution optimale $\hat{x}$ :**"))
display(ls.x_hat)

display(Markdown(r"- **Projection $\hat{y} = A \hat{x}$ :**"))
display(ls.y_hat)

display(Markdown(r"- **Résidu $\hat{r} = A \hat{x} - b$ :**"))
display(ls.r_hat)

display(Markdown(f"- **Norme du résidu $\\|\\hat{{r}}\\|$ :** ${sp.latex(ls.residual_norm)}$"))
display(Markdown(f"- **RMS du résidu $\\mathrm{{rms}}(\\hat{{r}})$ :** ${sp.latex(ls.residual_rms)}$"))

display(Markdown(r"### Contrôle d'orthogonalité du résidu ($A^T \hat{r} = 0$) :"))
display(ls.verify())

### Solution des Moindres Carrés :

- **Solution optimale $\hat{x}$ :**

Vector([9/10, 9/10])

- **Projection $\hat{y} = A \hat{x}$ :**

Vector([9/10, 9/5, 27/10, 18/5])

...

Vector([-1/10, -1/5, 7/10, -2/5])

- **Norme du résidu $\|\hat{r}\|$ :** $\frac{\sqrt{70}}{10}$

- **RMS du résidu $\mathrm{rms}(\hat{r})$ :** $\frac{\sqrt{70}}{20}$

### Contrôle d'orthogonalité du résidu ($A^T \hat{r} = 0$) :

- Vérification de la condition d'orthogonalité du résidu : A^T r_hat = 0 colonne par colonne.
  - Colonne a_1 : a_1^T r_hat = 0 (verdict : ZERO).
  - Colonne a_2 : a_2^T r_hat = 0 (verdict : ZERO).

## 9. Moteurs de Rendu et Traçabilité (`Explanation`)

Tout résultat algorithmique fournit `.explain()`, renvoyant une instance de `Explanation`
capable d'exporter le raisonnement dans différents formats.

In [19]:
exp = ls.explain(level=1)

display(Markdown("### Rendu interactif (Markdown + LaTeX) :"))
display(exp)

display(Markdown("### Extrait du code source LaTeX exportable :"))
print(exp.to_latex()[:350] + "\n...")

### Rendu interactif (Markdown + LaTeX) :

- Résolution au sens des moindres carrés min ||A x - b|| (A de taille 4×2, méthode 'rational_qr').
  - Vérification du rang plein en colonnes : rang(A) = 2 = 2.
  - Équations réduites par QR rationnelle : R_tilde x_hat = D^(-2) Q_tilde^T b = ( 9/4, 9/10 ).
    $$\left[\begin{array}{cc} 1 & \frac{3}{2} \\ 0 & 1 \end{array}\right]$$
  - Résolution par substitution arrière de R x = c (taille 2×2).
    $$\left[\begin{array}{cc} 1 & \frac{3}{2} \\ 0 & 1 \end{array}\right]$$
    - Calcul de x_2 : x_2 = (9/10 - (0)) / (1) = 9/10.
    - Calcul de x_1 : x_1 = (9/4 - (27/20)) / (1) = 9/10.
    - Solution du système triangulaire : x = ( 9/10, 9/10 ).
      $$\left[\begin{array}{c} \frac{9}{10} \\ \frac{9}{10} \end{array}\right]$$
  - Analyse de la solution : x_hat = ( 9/10, 9/10 ), projection y_hat = A x_hat = ( 9/10, 9/5, 27/10, 18/5 ), résidu r_hat = A x_hat - b = ( −1/10, −1/5, 7/10, −2/5 ).
    $$\left[\begin{array}{c} \frac{9}{10} \\ \frac{9}{10} \end{array}\right]$$
  - Norme du résidu ||r_hat|| = sqrt(70)/10, moyenne quadratique rms(r_hat) = sqrt(70)/20.

### Extrait du code source LaTeX exportable :

\begin{itemize}
  \item Résolution au sens des moindres carrés min ||A x - b|| (A de taille 4×2, méthode 'rational_qr').
  \begin{itemize}
    \item Vérification du rang plein en colonnes : rang(A) = 2 = 2.
    \item Équations réduites par QR rationnelle : R_tilde x_hat = D^(-2) Q_tilde^T b = ( 9/4, 9/10 ).
    \[\left[\begin{array}{cc} 1 & \frac{3
...


## 10. Exercices Interactifs et Détection des Erreurs Fréquentes

L'élève peut être guidé dans la réduction pas à pas d'un système. Le vérificateur d'opérations
analyse les propositions et détecte les erreurs classiques répertoriées dans l'Annexe C.

In [20]:
A_inter = Matrix([[1, 2], [3, 5]])
b_inter = Vector([5, 13])

session = GaussJordan(A_inter, b_inter).interactive()

# 1. Proposition valide et utile
fb1 = session.propose("L2 <- L2 - 3*L1")
display(Markdown(f"**Proposition 1 (`L2 <- L2 - 3*L1`) :** {fb1.message}"))

# 2. Détection d'erreur classique : dilatation par 0
fb_err = session.propose("L2 <- 0*L2")
display(Markdown(f"**Proposition 2 (`L2 <- 0*L2`) :** <span style='color:red;'>{fb_err.message}</span>"))

# 3. Demande de conseil sans donner la solution
display(Markdown(f"**Conseil du tuteur :** *{session.hint()}*"))

**Proposition 1 (`L2 <- L2 - 3*L1`) :** Valide. Le coefficient (2,1) est maintenant nul : progression vers la forme échelonnée.

**Proposition 2 (`L2 <- 0*L2`) :** <span style='color:red;'>Invalide : une dilatation par 0 n'est pas une opération élémentaire (perte d'information) ; l'opération n'est pas inversible : le système change.</span>

**Conseil du tuteur :** *Suggestion : annulez le coefficient (1,2) = 2 à l'aide de la ligne pivot L2.*

## 11. Comparaison Exact vs Flottant (`compare`)

Le module `compare` met en lumière les défaillances numériques du calcul en virgule flottante
sur les problèmes mal conditionnés de VMLS.

In [21]:
from linalg_pedagopy.compare import (
    lauchli_gram_schmidt_experiment,
    lauchli_least_squares_experiment,
    hilbert_conditioning_experiment
)

# 1. Perte d'orthogonalité sur la matrice de Läuchli (eps = 1e-8)
display(Markdown(f"```text\n{lauchli_gram_schmidt_experiment()}\n```"))

# 2. Échec des équations normales en flottant
display(Markdown(f"```text\n{lauchli_least_squares_experiment()}\n```"))

```text
Matrice de Läuchli (4x3, ε = 1.0e-08) :
- Exact (SymPy)              : max |q_i^T q_j - δ_ij| = 0
- Gram-Schmidt classique     : max |q_i^T q_j - δ_ij| = 0.5000 (perte totale d'orthogonalité)
- Gram-Schmidt modifié (MGS) : max |q_i^T q_j - δ_ij| = 7.07e-09
```

```text
Moindres carrés sur Läuchli (ε = 1.0e-08) :
- En flottant, 1 + ε^2 == 1 : A^T A est arrondie en une matrice de 1 (rang 1 < 3, singulière !).
- Les équations normales échouent en flottant, tandis que QR réussit (erreur relative = 4.53e-16).
```

## 12. Le Tuteur Interactif pour le Recueil de VMLS (`tuteur`)

Le module `tuteur` permet de s'entraîner sur les 136 exercices du recueil officiel de VMLS
(*Additional Exercises for VMLS*, Boyd & Vandenberghe).

### Les 8 principes pédagogiques :
1. **Vérification par propriété** : accepte toute réponse mathématiquement valide.
2. **Diagnostic explicatif** : montre ce que calcule la réponse de l'étudiant sans afficher la cible.
3. **Indices gradués sur 4 niveaux** (Orientation $\to$ Outil $\to$ Démarrage $\to$ Exercice jumeau résolu).
4. **Exercice jumeau résolu** au niveau 4 pour transposer la méthode.
5. **Verrou des données** : `ex.donnees` bloque les calculs automatiques tant qu'aucune réponse n'a été soumise.
6. **Engagement avant exploration** : prédiction obligatoire (`ex.predire(...)`) pour ouvrir le laboratoire.
7. **Refus de confirmer par de simples essais** : un essai numérique ne vaut pas preuve.
8. **Journal local transparent** : historique et synthèse via `ex.bilan()`.

## 13. Exercices Pratiques Pas à Pas

Résolvons maintenant 5 exercices représentatifs du recueil VMLS avec le tuteur.

### Exercice Pratique 1 : Modélisation d'une moyenne glissante (Recueil VMLS 6.4)

**Énoncé** :
$p \in \mathbb{R}^6$ est une série temporelle de prix sur 6 jours.
$d \in \mathbb{R}^3$ mesure, à partir du 4e jour ($i = 1, 2, 3$), l'écart entre le prix du jour ($p_{i+3}$)
et la moyenne des 3 jours précédents :
$$d_i = p_{i+3} - \frac{p_i + p_{i+1} + p_{i+2}}{3}$$
Trouver la matrice $A$ telle que $d = A p$.

In [22]:
ex64 = exercice("6.4")
display(Markdown(f"**Énoncé :**\n>{ex64.enonce()}\n\n**Indice 1 :** {ex64.indice()}"))

# Test avec une matrice contenant une erreur sur la ligne 2 :
t = Fraction(1, 3)
A_err = Matrix([
    [-t, -t, -t, 1, 0, 0],
    [-t, -t, -t, 1, 0, 0],  # Erreur : recopie de la ligne 1
    [0, 0, -t, -t, -t, 1]
])
diag_err = ex64.verifier(A_err)
display(Markdown(f"**Diagnostic sur réponse erronée :**\n> {diag_err}"))

# Soumission de la matrice exacte :
A_ok = Matrix([
    [-t, -t, -t, 1, 0, 0],
    [0, -t, -t, -t, 1, 0],
    [0, 0, -t, -t, -t, 1]
])
diag_ok = ex64.verifier(A_ok)
display(Markdown(f"**Validation de la matrice exacte :**\n> {diag_ok}"))
display(Markdown(f"```text\n{ex64.bilan()}\n```"))

**Énoncé :**
>[Exercice 6.4 — Recueil Additional Exercises de VMLS, page 15]
p ∈ R^6 est une série de prix. d ∈ R^3 mesure, à partir du 4e jour, l'écart entre le prix du jour et la moyenne des trois jours précédents. Trouver la matrice A telle que d = A p (donner sa taille et ses coefficients).
Notions : matrice comme application linéaire, VMLS ch. 6

**Indice 1 :** [Niveau 1 — Orientation]
Quelle est la taille de d ? celle de p ? Qu'en déduisez-vous pour A ?

**Diagnostic sur réponse erronée :**
> Taille correcte (3 x 6). Lignes 1 et 3 correctes.
Votre ligne 2 calcule d_2 = p_4 - (p_1 + p_2 + p_3)/3.
Relisez la définition de d_2 : quel est le prix « du jour » pour i = 2 ?

**Validation de la matrice exacte :**
> Taille correcte (3 x 6). Toutes les lignes de votre matrice calculent exactement d = A p.

```text
=== Bilan de l'exercice 6.4 ===
- Statut               : Validé
- Tentatives soumises  : 2
- Indices consultés    : 1
- Exercice jumeau lu   : non
- Essais au labo       : 0
- Temps passé          : 0.2 s
```

### Exercice Pratique 2 : Vrai/Faux au Laboratoire de Contre-Exemples (Recueil VMLS 3.3)

**Énoncé** :
Soient $a, b, c \in \mathbb{R}^n$ tels que $a \perp c$ et $b \perp c$.
(a) A-t-on toujours $a \perp b$ ?
(b) A-t-on toujours $(a + b) \perp c$ ?
(c) A-t-on toujours $(a - b) \perp (a + b)$ ?

In [23]:
ex33 = exercice("3.3")

# Étape 1 : S'engager en enregistrant sa prédiction
ex33.predire(a="peut être faux", b="toujours vrai", c="peut être faux")

# Étape 2 : Ouvrir le laboratoire et tester une instance dans R^3
lab = ex33.labo()
rep = lab.tester(
    a=Vector([1, 0, 0]),
    b=Vector([1, 1, 0]),
    c=Vector([0, 0, 1])
)

display(Markdown(f"**Résultat du test au laboratoire :**\n\n{rep.message}"))

**Résultat du test au laboratoire :**

Hypothèses vérifiées : a ⊥ c et b ⊥ c.
(a) est faux sur votre instance : c'est un contre-exemple valide.
(b) est vrai sur votre instance.
(c) est faux sur votre instance : c'est un contre-exemple valide.
Pour (b), des essais ne suffisent pas : rédigez une justification.

### Exercice Pratique 3 : Recherche de deux Inverses à Droite (Recueil VMLS 11.11)

**Énoncé** :
Pour la matrice large $A = \begin{pmatrix} 1 & -1 & 2 \\ 0 & 1 & 1 \end{pmatrix}$,
trouver deux matrices inverses à droite distinctes $B_1$ et $B_2$ ($3 \times 2$)
telles que $A B_1 = I_2$, $A B_2 = I_2$ et $B_1 \neq B_2$.

In [24]:
ex1111 = exercice("11.11")

# Construction de deux inverses à droite distinctes
B1 = Matrix([[1, 1], [0, 1], [0, 0]])
B2 = Matrix([[-2, 1], [-1, 1], [1, 0]])

display(Markdown(r"**Matrice $B_1$ :**"))
display(B1)
display(Markdown(r"**Matrice $B_2$ :**"))
display(B2)

# Soumission du couple (B1, B2)
resultat = ex1111.verifier((B1, B2))
display(Markdown(f"**Diagnostic :** {resultat.diagnostic}"))

**Matrice $B_1$ :**

Matrix([[1, 1], [0, 1], [0, 0]])

**Matrice $B_2$ :**

Matrix([[-2, 1], [-1, 1], [1, 0]])

**Diagnostic :** Bravo ! A B1 = I_2, A B2 = I_2 et B1 ≠ B2 : vos deux inverses à droite sont valides.

### Exercice Pratique 4 : Détecteurs d'Erreurs Typiques et Verrou (Recueil VMLS 3.9)

Cet exercice illustre le **verrou de données** (`ex.donnees`) et les **détecteurs d'erreurs typiques**
(norme au lieu de RMS, écart-type en $n-1$, carré de la distance, angle au lieu de corrélation).

In [25]:
ex39 = exercice("3.9")

# 1. Tentative de calcul direct sur les données verrouillées :
try:
    ex39.donnees.x.rms()
except Exception as e:
    display(Markdown(f"**Tentative interceptée par le verrou :** `{type(e).__name__}`"))

x = Vector([1, -2, 3, 2])

# 2. Erreur typique 1 : soumettre ||x|| au lieu de rms(x)
diag_e1 = ex39.verifier(x.norm()).diagnostic
display(Markdown(f"**Test Erreur typique 1 :** {diag_e1}"))

# 3. Soumission de la bonne valeur : déverrouille ex.donnees.x !
diag_ok = ex39.verifier(x.rms()).diagnostic
display(Markdown(f"**Soumission exacte :** {diag_ok}"))
display(Markdown(f"- Données déverrouillées, calcul direct autorisé : **${sp.latex(ex39.donnees.x.rms())}$**"))

**Tentative interceptée par le verrou :** `LockedExerciseDataError`

**Test Erreur typique 1 :** Votre valeur est la norme. Que faut-il encore faire pour obtenir une moyenne quadratique ?

**Soumission exacte :** Exact ! Votre valeur est correcte.

- Données déverrouillées, calcul direct autorisé : **$\frac{3 \sqrt{2}}{2}$**

### Exercice Pratique 5 : Optimisation de Norme Minimale et Système KKT (Recueil VMLS 20.11)

**Énoncé** :
Trouver le vecteur $x \in \mathbb{R}^3$ minimisant $\|x\|^2$ sous la contrainte $C x = d$ avec :
$$C = \begin{pmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \end{pmatrix}, \quad d = \begin{pmatrix} 1 \\ 1 \end{pmatrix}$$

In [26]:
ex2011 = exercice("20.11")
display(Markdown(f"**Énoncé :**\n>{ex2011.enonce()}"))

# Solution analytique du système KKT : x* = C^T (C C^T)^(-1) d
x_opt = Vector([Fraction(1, 3), Fraction(2, 3), Fraction(1, 3)])

display(Markdown(r"**Vecteur optimal $x^*$ :**"))
display(x_opt)

res_kkt = ex2011.verifier(x_opt)
display(Markdown(f"**Diagnostic :** {res_kkt.diagnostic}"))

**Énoncé :**
>[Exercice 20.11 — Recueil Additional Exercises de VMLS, page 49]
On considère le problème de norme minimale : minimiser ||x||^2 sous la contrainte C x = d, avec C = [[1, 1, 0], [0, 1, 1]] (taille 2×3) et d = (1, 1). Trouver la solution optimale x de R^3 à l'aide des conditions KKT.
Notions : solution de norme minimale, système KKT, VMLS ch. 16 et 20

**Vecteur optimal $x^*$ :**

Vector([1/3, 2/3, 1/3])

**Diagnostic :** Exact ! Votre vecteur x vérifie C x = d et les conditions KKT de norme minimale.

## 14. Glossaire Anglais-Français Intégré

La fonction `glossaire(terme)` permet d'obtenir la définition et la traduction française d'un concept clé de VMLS.

In [27]:
mots = ("least squares", "root-mean-square", "de-meaned vector", "running average", "gram matrix")
lignes = [f"- **{glossaire(m)}**" for m in mots]
display(Markdown("### Extraits du Glossaire bilingue VMLS :\n" + "\n".join(lignes)))

### Extraits du Glossaire bilingue VMLS :
- **least squares → Moindres carrés : Problème d'optimisation consistant à chercher x_hat qui minimise ||A x - b||² (ou ||A x - b||). La condition d'optimalité est l'orthogonalité du résidu : A^T (A x_hat - b) = 0 (VMLS ch. 12).**
- **root-mean-square → Moyenne quadratique (rms) : Pour un vecteur x de taille n, rms(x) = sqrt((x_1² + ... + x_n²) / n) = ||x|| / sqrt(n) (VMLS ch. 1 et 3).**
- **de-meaned vector → Vecteur centré : Vecteur obtenu en retranchant la moyenne arithmétique à chaque composante : x_tilde = x - avg(x)·1 (VMLS ch. 3).**
- **running average → Moyenne glissante (ou mobile) : Moyenne calculée sur une fenêtre fixe des k dernières valeurs d'une série temporelle (VMLS ch. 6).**
- **gram matrix → Matrice de Gram : Matrice symétrique G = A^T A dont le coefficient (i, j) est le produit scalaire a_i^T a_j des colonnes i et j de A (VMLS ch. 10).**